In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
N = 100000  # 100K synthetic transactions

print("Generating synthetic transaction dataset...")

Generating synthetic transaction dataset...


In [2]:
# Generate base transaction features
def generate_transactions(n):
    data = []
    
    for i in range(n):
        # Is this transaction fraud? (1.5% fraud rate, realistic)
        is_fraud = np.random.random() < 0.015
        
        if is_fraud:
            # Fraud patterns — biased toward suspicious combinations
            amount = np.random.choice([
                np.random.uniform(5000, 50000),   # large amounts
                np.random.uniform(0.01, 1.00),    # micro-testing
                np.random.uniform(100, 500)
            ], p=[0.5, 0.3, 0.2])
            
            merchant = np.random.choice(
                ["Crypto exchange", "ATM withdrawal", "Online transfer", "Luxury goods", "Electronics", "Travel"],
                p=[0.30, 0.25, 0.20, 0.10, 0.10, 0.05]
            )
            location = np.random.choice(
                ["High-risk country", "International", "Different state", "Same city"],
                p=[0.45, 0.30, 0.20, 0.05]
            )
            time_of_day = np.random.choice(
                ["Late night (12–5am)", "Early morning", "Evening", "Business hours"],
                p=[0.45, 0.25, 0.20, 0.10]
            )
            tx_24h = np.random.randint(5, 20)
            card_age = np.random.randint(0, 12)
            device = np.random.choice(
                ["VPN / proxy detected", "New device, new IP", "New device, matching IP", "Known device, new IP", "Known device, matching IP"],
                p=[0.35, 0.30, 0.15, 0.15, 0.05]
            )
        else:
            # Legitimate patterns
            amount = np.random.choice([
                np.random.uniform(5, 200),
                np.random.uniform(200, 2000),
                np.random.uniform(2000, 8000)
            ], p=[0.60, 0.30, 0.10])
            
            merchant = np.random.choice(
                ["Groceries", "Restaurant", "Electronics", "Travel", "Online transfer", "ATM withdrawal", "Luxury goods", "Crypto exchange"],
                p=[0.30, 0.25, 0.15, 0.10, 0.08, 0.06, 0.04, 0.02]
            )
            location = np.random.choice(
                ["Same city", "Different state", "International", "High-risk country"],
                p=[0.70, 0.20, 0.08, 0.02]
            )
            time_of_day = np.random.choice(
                ["Business hours", "Evening", "Early morning", "Late night (12–5am)"],
                p=[0.50, 0.30, 0.12, 0.08]
            )
            tx_24h = np.random.randint(0, 6)
            card_age = np.random.randint(6, 120)
            device = np.random.choice(
                ["Known device, matching IP", "Known device, new IP", "New device, matching IP", "New device, new IP", "VPN / proxy detected"],
                p=[0.70, 0.15, 0.08, 0.05, 0.02]
            )
        
        data.append({
            "amount": round(amount, 2),
            "merchant_category": merchant,
            "location": location,
            "time_of_day": time_of_day,
            "transactions_last_24h": tx_24h,
            "card_age_months": card_age,
            "device_match": device,
            "is_fraud": int(is_fraud)
        })
    
    return pd.DataFrame(data)

df = generate_transactions(N)
print(f"Dataset shape: {df.shape}")
print(f"\nFraud distribution:")
print(df['is_fraud'].value_counts())
print(f"\nFraud rate: {df['is_fraud'].mean()*100:.2f}%")
df.head()

Dataset shape: (100000, 8)

Fraud distribution:
is_fraud
0    98474
1     1526
Name: count, dtype: int64

Fraud rate: 1.53%


,amount,merchant_category,location,time_of_day,transactions_last_24h,card_age_months,device_match,is_fraud
0,190.39,Groceries,Same city,Early morning,3,109,"Known device, new IP",0
1,194.13,Groceries,Same city,Evening,3,94,"Known device, matching IP",0
2,32.20,Travel,Same city,Evening,0,8,"New device, matching IP",0
3,92.85,Restaurant,Same city,Business hours,3,116,"Known device, matching IP",0
4,167.47,Travel,Same city,Business hours,3,59,"Known device, matching IP",0


In [3]:
# Feature engineering
df_encoded = df.copy()

# Binary flags
df_encoded['is_high_risk_location'] = df_encoded['location'].isin(['High-risk country', 'International']).astype(int)
df_encoded['is_late_night'] = df_encoded['time_of_day'].isin(['Late night (12–5am)', 'Early morning']).astype(int)
df_encoded['is_vpn'] = df_encoded['device_match'].isin(['VPN / proxy detected']).astype(int)
df_encoded['is_new_device'] = df_encoded['device_match'].isin(['New device, new IP', 'New device, matching IP']).astype(int)
df_encoded['is_high_risk_merchant'] = df_encoded['merchant_category'].isin(['Crypto exchange', 'ATM withdrawal', 'Online transfer']).astype(int)
df_encoded['is_new_card'] = (df_encoded['card_age_months'] < 6).astype(int)
df_encoded['is_high_amount'] = (df_encoded['amount'] > 5000).astype(int)
df_encoded['is_micro_transaction'] = (df_encoded['amount'] < 1.0).astype(int)
df_encoded['is_high_velocity'] = (df_encoded['transactions_last_24h'] > 5).astype(int)

# Interaction features
df_encoded['amount_per_card_age'] = df_encoded['amount'] / (df_encoded['card_age_months'] + 1)
df_encoded['velocity_x_amount'] = df_encoded['transactions_last_24h'] * df_encoded['amount']
df_encoded['risk_score'] = (
    df_encoded['is_high_risk_location'] * 3 +
    df_encoded['is_vpn'] * 3 +
    df_encoded['is_late_night'] * 2 +
    df_encoded['is_new_device'] * 2 +
    df_encoded['is_high_risk_merchant'] * 2 +
    df_encoded['is_new_card'] * 2 +
    df_encoded['is_high_amount'] * 2 +
    df_encoded['is_high_velocity'] * 2 +
    df_encoded['is_micro_transaction'] * 2
)

# Label encode categoricals
le_merchant = LabelEncoder()
le_location = LabelEncoder()
le_time = LabelEncoder()
le_device = LabelEncoder()

df_encoded['merchant_enc'] = le_merchant.fit_transform(df_encoded['merchant_category'])
df_encoded['location_enc'] = le_location.fit_transform(df_encoded['location'])
df_encoded['time_enc'] = le_time.fit_transform(df_encoded['time_of_day'])
df_encoded['device_enc'] = le_device.fit_transform(df_encoded['device_match'])

# Scale amount
scaler = StandardScaler()
df_encoded['amount_scaled'] = scaler.fit_transform(df_encoded[['amount']])

# Final feature set
feature_cols = [
    'amount_scaled', 'transactions_last_24h', 'card_age_months',
    'merchant_enc', 'location_enc', 'time_enc', 'device_enc',
    'is_high_risk_location', 'is_late_night', 'is_vpn', 'is_new_device',
    'is_high_risk_merchant', 'is_new_card', 'is_high_amount',
    'is_micro_transaction', 'is_high_velocity',
    'amount_per_card_age', 'velocity_x_amount', 'risk_score'
]

X = df_encoded[feature_cols]
y = df_encoded['is_fraud']

print(f"Features: {len(feature_cols)}")
print(f"Feature names: {feature_cols}")
print(f"\nClass distribution:")
print(y.value_counts())

Features: 19
Feature names: ['amount_scaled', 'transactions_last_24h', 'card_age_months', 'merchant_enc', 'location_enc', 'time_enc', 'device_enc', 'is_high_risk_location', 'is_late_night', 'is_vpn', 'is_new_device', 'is_high_risk_merchant', 'is_new_card', 'is_high_amount', 'is_micro_transaction', 'is_high_velocity', 'amount_per_card_age', 'velocity_x_amount', 'risk_score']

Class distribution:
is_fraud
0    98474
1     1526
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape}")
print(f"Test size: {X_test.shape}")
print(f"\nBefore SMOTE: {y_train.value_counts().to_dict()}")

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE: {pd.Series(y_train_res).value_counts().to_dict()}")
print(f"Total training samples: {len(X_train_res)}")

Train size: (80000, 19)
Test size: (20000, 19)

Before SMOTE: {0: 78779, 1: 1221}

After SMOTE: {0: 78779, 1: 78779}
Total training samples: 157558


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.ensemble import IsolationForest
from xgboost import XGBClassifier
import time

models = {}

print("Training Logistic Regression...")
t = time.time()
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_res, y_train_res)
models["logistic_regression"] = lr
print(f"✅ Done in {time.time()-t:.1f}s")

print("\nTraining Random Forest...")
t = time.time()
rf = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train_res, y_train_res)
models["random_forest"] = rf
print(f"✅ Done in {time.time()-t:.1f}s")

print("\nTraining XGBoost...")
t = time.time()
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)
xgb.fit(X_train_res, y_train_res)
models["xgboost"] = xgb
print(f"✅ Done in {time.time()-t:.1f}s")

print("\nTraining Gradient Boosting...")
t = time.time()
gb = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    random_state=42
)
gb.fit(X_train_res, y_train_res)
models["gradient_boosting"] = gb
print(f"✅ Done in {time.time()-t:.1f}s")

print("\nTraining Isolation Forest...")
t = time.time()
iso = IsolationForest(n_estimators=200, contamination=0.015, random_state=42)
iso.fit(X_train)
models["isolation_forest"] = iso
print(f"✅ Done in {time.time()-t:.1f}s")

print("\n✅ All 5 models trained.")

Training Logistic Regression...
✅ Done in 0.8s

Training Random Forest...
✅ Done in 1.3s

Training XGBoost...
✅ Done in 0.6s

Training Gradient Boosting...
✅ Done in 51.6s

Training Isolation Forest...
✅ Done in 0.5s

✅ All 5 models trained.


In [6]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

results = {}

for name, model in models.items():
    if name == "isolation_forest":
        # Isolation forest doesn't have predict_proba
        y_pred_raw = model.predict(X_test)
        y_pred = (y_pred_raw == -1).astype(int)
        print(f"\n{'='*50}")
        print(f"  {name.upper()}")
        print(f"{'='*50}")
        print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))
        results[name] = {"auc": None, "report": classification_report(y_test, y_pred, output_dict=True)}
    else:
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
        report = classification_report(y_test, y_pred, target_names=['Legit', 'Fraud'], output_dict=True)
        results[name] = {"auc": round(auc, 4), "report": report}
        print(f"\n{'='*50}")
        print(f"  {name.upper()}")
        print(f"{'='*50}")
        print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))
        print(f"ROC-AUC: {auc:.4f}")

print("\n📊 SUMMARY TABLE")
print(f"{'Model':<25} {'Precision':>10} {'Recall':>10} {'F1':>10} {'AUC':>10}")
print("-" * 65)
for name, r in results.items():
    if name != "isolation_forest":
        p = r['report']['Fraud']['precision']
        rec = r['report']['Fraud']['recall']
        f1 = r['report']['Fraud']['f1-score']
        auc = r['auc']
        print(f"{name:<25} {p:>10.3f} {rec:>10.3f} {f1:>10.3f} {auc:>10.4f}")


  LOGISTIC_REGRESSION
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     19695
       Fraud       0.98      1.00      0.99       305

    accuracy                           1.00     20000
   macro avg       0.99      1.00      0.99     20000
weighted avg       1.00      1.00      1.00     20000

ROC-AUC: 1.0000

  RANDOM_FOREST
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     19695
       Fraud       1.00      1.00      1.00       305

    accuracy                           1.00     20000
   macro avg       1.00      1.00      1.00     20000
weighted avg       1.00      1.00      1.00     20000

ROC-AUC: 1.0000

  XGBOOST
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     19695
       Fraud       1.00      1.00      1.00       305

    accuracy                           1.00     20000
   macro avg       1.00      1.00      1.00  

In [7]:
import joblib
import os

os.makedirs('../artifacts', exist_ok=True)

# Save models
joblib.dump(models["xgboost"],            '../artifacts/xgb_v2.pkl')
joblib.dump(models["random_forest"],      '../artifacts/rf_v2.pkl')
joblib.dump(models["logistic_regression"],'../artifacts/lr_v2.pkl')
joblib.dump(models["gradient_boosting"],  '../artifacts/gb_v2.pkl')
joblib.dump(models["isolation_forest"],   '../artifacts/iso_v2.pkl')

# Save encoders and scaler — needed to transform API inputs
joblib.dump(le_merchant, '../artifacts/le_merchant.pkl')
joblib.dump(le_location, '../artifacts/le_location.pkl')
joblib.dump(le_time,     '../artifacts/le_time.pkl')
joblib.dump(le_device,   '../artifacts/le_device.pkl')
joblib.dump(scaler,      '../artifacts/scaler.pkl')

# Save feature columns list
import json
with open('../artifacts/feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)

# Save model metrics for the /metrics endpoint
metrics = {}
for name, r in results.items():
    if name != "isolation_forest":
        metrics[name] = {
            "precision": round(r['report']['Fraud']['precision'], 4),
            "recall":    round(r['report']['Fraud']['recall'], 4),
            "f1":        round(r['report']['Fraud']['f1-score'], 4),
            "auc":       r['auc']
        }

with open('../artifacts/model_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("✅ All models saved.")
print("✅ Encoders saved.")
print("✅ Metrics saved.")
print("\nFiles in artifacts/:")
for f in sorted(os.listdir('../artifacts')):
    print(f"  - {f}")

✅ All models saved.
✅ Encoders saved.
✅ Metrics saved.

Files in artifacts/:
  - confusion_matrix.png
  - feature_cols.json
  - feature_importance.png
  - fraud_model.pkl
  - gb_v2.pkl
  - iso_model.pkl
  - iso_v2.pkl
  - le_device.pkl
  - le_location.pkl
  - le_merchant.pkl
  - le_time.pkl
  - lr_model.pkl
  - lr_v2.pkl
  - model_metrics.json
  - rf_model.pkl
  - rf_v2.pkl
  - scaler.pkl
  - xgb_v2.pkl
